# PCBDetect: Exploratory Data Analysis (EDA) on DeepPCB
### Milestone 0: Data Foundation & Industrial Quality Control Formulation

This notebook conducts a deep-dive exploratory data analysis into the **DeepPCB** benchmark dataset, evaluating:
1. **Defect Class Distribution & Balance** across 6 critical manufacturing defect types.
2. **Bounding Box Dimensions & The Small Object Challenge** (COCO small regime).
3. **Board-Level Data Leakage Analysis** between default trainval and test splits.
4. **Classical Template Alignment & Difference Residuals** as an intuitive baseline.

In [ ]:
import sys
from pathlib import Path

import cv2

# Add project root to sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

from pcbdetect.constants import RAW_DATA_DIR
from pcbdetect.data.dataset import load_deeppcb_dataset
from pcbdetect.data.eda import run_eda

print(f"Project Root: {ROOT_DIR}")
print(f"Raw Data Dir exists: {RAW_DATA_DIR.exists()}")

## 1. Load Dataset & Summary Statistics
We load all 1,500 paired images (1,000 trainval + 500 test) and parse the bounding boxes.

In [ ]:
samples = load_deeppcb_dataset(split="all")
print(f"Loaded {len(samples)} image pairs.")

# Sample preview
s0 = samples[0]
print(f"Sample ID: {s0.sample_id}")
print(f"Board Group: {s0.group_id}")
print(f"Split: {s0.split}")
print(f"Number of defects: {s0.num_defects}")
for b in s0.boxes:
    print(f"  - [{b.class_name}] coords: ({b.x1}, {b.y1}) to ({b.x2}, {b.y2}) | Area: {b.area} px2")

## 2. Execute Automated Statistical EDA Pipeline
Compute class frequencies, bounding box geometry, and check for cross-split group leakage.

In [ ]:
stats = run_eda(save_figures=True)
print("\n--- Overall Dataset Statistics ---")
for k, v in stats.items():
    if isinstance(v, dict):
        print(f"{k}: {list(v.keys())}")
    else:
        print(f"{k}: {v}")

## 3. Visualizing Defects & Classical Template Matching
Compare the defect-free golden template, the tested PCB, ground truth bounding boxes, and the pixel-wise difference image.

In [ ]:
# Pick a sample with multiple diverse defects
sample = next((s for s in samples if s.num_defects >= 6), samples[0])
temp_img = sample.load_template_image()
test_img = sample.load_test_image()
overlay = sample.render_overlay(thickness=2)
diff = cv2.absdiff(test_img, temp_img)

# Display using OpenCV or saved publication figures
fig_path = ROOT_DIR / "docs" / "figures" / "sample_defect_visualizations.png"
print(f"Visualization saved to: {fig_path}")

## 4. Key Takeaways for Model Architecture (Milestone 1 - 3)
- **32% of defects are COCO Small (< 32x32 pixels)**, and 68% are medium. Detectors with aggressive feature stride (P5 32x) lose spatial resolution; P2/P3 strides or feature pyramids (FPN/PANet) are necessary.
- **7 overlapping board groups** between default `trainval` and `test` splits create potential data leakage if evaluated purely on random splits. In Milestone 1, we implement strict group-disjoint evaluation.
- **Recall SLA prioritization**: In factory economics, a False Negative (escaped short/open) costs $50-$5000+ in warranty and board scrap, while a False Positive costs <$0.50 in operator review time.